## Step 1: Cohort Selection

In [1]:
# Import required library 
import pandas as pd

In [2]:
# load raw static data
df_static = pd.read_csv("../data/MIMIC-IV-static(Group Assignment).csv")
print(f"Raw data: {df_static.shape}")

Raw data: (76943, 142)


In [3]:
# exclude non-ICU units
EXCLUDED_CAREUNITS = ["Neuro Intermediate", "Neuro Stepdown"]
df = df_static[~df_static["first_careunit"].isin(EXCLUDED_CAREUNITS)]
print(f"After excluding non-ICU units: {len(df)} ({len(df_static) - len(df)} removed)")

After excluding non-ICU units: 73731 (3212 removed)


In [4]:
# keep first ICU stay per patient (by intime)
df["intime"] = pd.to_datetime(df["intime"])
df = df.sort_values("intime").groupby("subject_id").first().reset_index()
print(f"After keeping first stay per patient: {len(df)}")

/tmp/ipykernel_271884/690109032.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["intime"] = pd.to_datetime(df["intime"])


After keeping first stay per patient: 51333


In [5]:
# require ICU LOS > 24 hours
df = df[df["icu_los_hours"] > 24]
print(f"After requiring LOS > 24h: {len(df)}")

After requiring LOS > 24h: 40599


In [6]:
# exclude patients who died before 24 hours
df["deathtime"] = pd.to_datetime(df["deathtime"])
hours_to_death = (df["deathtime"] - df["intime"]).dt.total_seconds() / 3600
died_before_24 = hours_to_death.lt(24).fillna(False)
df = df[~died_before_24]
print(f"After excluding deaths before 24h: {len(df)}")

After excluding deaths before 24h: 40457


In [7]:
# save cohort
df.to_csv("cohort.csv", index=False)
print(f"Saved cohort.csv: {df.shape}")

Saved cohort.csv: (40457, 142)
